# Titanic Data Project — Cleaning, Analysis & KPI Extraction
**Python only** (pandas / NumPy). This notebook:
1. Loads the raw file and audits its data quality
2. Cleans and imputes missing values properly (not just `dropna`)
3. Engineers features used both for imputation and analysis
4. Extracts KPIs and insights
5. Exports `titanic_cleaned.csv` and `kpis.json`, which power the interactive HTML dashboard


In [ ]:
import json, re
import numpy as np
import pandas as pd

RAW = "File_3.csv"                 # <-- change path if needed
OUT_CSV = "titanic_cleaned.csv"
OUT_JSON = "kpis.json"

pd.set_option("display.max_columns", None)

## 1. Load & Data Quality Audit
Check shape, duplicates, and missing values **before** touching anything.

In [ ]:
df = pd.read_csv(RAW)
print(df.shape)
df.head()

In [ ]:
audit = {
    "rows": int(df.shape[0]),
    "cols": int(df.shape[1]),
    "duplicated_rows": int(df.duplicated().sum()),
    "duplicated_ids": int(df["PassengerId"].duplicated().sum()),
    "missing_before": {k: int(v) for k, v in df.isna().sum().items() if v > 0},
    "missing_pct_before": {k: round(v * 100 / len(df), 2)
                           for k, v in df.isna().mean().items() if v > 0},
}
audit

## 2. Cleaning
Trim whitespace, normalise empty strings to `NaN`, and extract the passenger **Title** from `Name` — this becomes a key feature for smart age imputation.

In [ ]:
df.columns = [c.strip() for c in df.columns]
for c in df.select_dtypes("object"):
    df[c] = df[c].astype(str).str.strip().replace({"nan": np.nan, "": np.nan})

df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")[0].str.strip()
title_map = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs", "Lady": "Rare", "Sir": "Rare",
             "Capt": "Rare", "Col": "Rare", "Major": "Rare", "Don": "Rare", "Dona": "Rare",
             "Jonkheer": "Rare", "Countess": "Rare", "the Countess": "Rare", "Dr": "Rare", "Rev": "Rare"}
df["Title"] = df["Title"].replace(title_map).fillna("Rare")
df["Title"].value_counts()

## 3. Missing Values
Each column gets the treatment that fits its distribution and role — never a blanket `fillna(0)`.

**Age** (86 missing, 20.6%) → median of the same `Title` + `Pclass` group, falling back to class median, then overall median.

In [ ]:
df["Age"] = df.groupby(["Title", "Pclass"])["Age"].transform(lambda s: s.fillna(s.median()))
df["Age"] = df.groupby("Pclass")["Age"].transform(lambda s: s.fillna(s.median()))
df["Age"] = df["Age"].fillna(df["Age"].median())

**Fare** (1 missing, plus a handful of invalid zeros) → median fare *within the same class*, since fare is class-driven and right-skewed (median beats mean here).

In [ ]:
df["Fare"] = df.groupby("Pclass")["Fare"].transform(lambda s: s.fillna(s.median()))
df.loc[df["Fare"] <= 0, "Fare"] = np.nan   # a £0 fare is treated as invalid, not a real ticket price
df["Fare"] = df.groupby("Pclass")["Fare"].transform(lambda s: s.fillna(s.median()))

**Embarked** (a few missing) → simple mode imputation, appropriate for a low-cardinality categorical column.

In [ ]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

**Cabin** (327 missing, 78.2%) → too sparse to impute meaningfully. Instead of dropping the signal entirely, extract the **Deck** letter and a **HasCabin** flag, then drop the raw column.

In [ ]:
df["Deck"] = df["Cabin"].str[0].fillna("Unknown")
df["HasCabin"] = df["Cabin"].notna().astype(int)
df = df.drop(columns=["Cabin"])

print("Remaining missing values:", df.isna().sum().sum())

## 4. Outliers
Fare is right-skewed with a long tail of genuine first-class fares. Rather than deleting them, flag them with the IQR rule and add a **capped** copy for plotting.

In [ ]:
q1, q3 = df["Fare"].quantile([.25, .75]); iqr = q3 - q1
upper = q3 + 1.5 * iqr
n_out = int((df["Fare"] > upper).sum())
df["Fare_capped"] = df["Fare"].clip(upper=upper)
print(f"{n_out} fares above the IQR fence of £{upper:.2f}")

## 5. Feature Engineering
Build the fields used throughout the analysis and the dashboard.

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df["AgeGroup"] = pd.cut(df["Age"], [0, 12, 18, 35, 60, 100],
                        labels=["Child (0-12)", "Teen (13-18)", "Young Adult (19-35)",
                                "Adult (36-60)", "Senior (60+)"])
df["FareBand"] = pd.qcut(df["Fare"], 4, labels=["Low", "Medium", "High", "Very High"])
df["Pclass_name"] = df["Pclass"].map({1: "1st Class", 2: "2nd Class", 3: "3rd Class"})
df["Embarked_name"] = df["Embarked"].map({"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"})
df["Status"] = df["Survived"].map({1: "Survived", 0: "Died"})
df["TicketGroup"] = df.groupby("Ticket")["Ticket"].transform("size")

df.to_csv(OUT_CSV, index=False)
df.head()

## 6. Exploratory Visuals (Matplotlib / Seaborn)
Static EDA charts — quick to render, good for a report or notebook read-through. The full *interactive* exploration lives in the Plotly/Streamlit dashboard (`app_streamlit.py`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.countplot(data=df, x="Pclass_name", hue="Status", ax=axes[0,0])
axes[0,0].set_title("Survival count by class")
sns.histplot(data=df, x="Age", hue="Status", multiple="stack", bins=20, ax=axes[0,1])
axes[0,1].set_title("Age distribution by outcome")
sns.boxplot(data=df, x="Pclass_name", y="Fare_capped", ax=axes[1,0])
axes[1,0].set_title("Fare by class (capped)")
corr_cols = ["Survived","Pclass","Age","Fare","FamilySize","IsAlone","HasCabin"]
sns.heatmap(df[corr_cols].corr(), annot=True, cmap="coolwarm", center=0, ax=axes[1,1])
axes[1,1].set_title("Correlation heatmap")
plt.tight_layout()
plt.savefig("eda_overview.png", dpi=120)
plt.show()

## 7. KPIs & Insights
Aggregate the cleaned data into the numbers that drive the dashboard.

In [ ]:
r = lambda x: round(float(x), 2)
sr = lambda g: (df.groupby(g)["Survived"].mean() * 100).round(2)
cnt = lambda g: df.groupby(g)["Survived"].size()

k = {
    "audit": audit,
    "missing_after": {k2: int(v) for k2, v in df.isna().sum().items() if v > 0},
    "outliers_fare": n_out, "fare_upper": r(upper),
    "total": int(len(df)),
    "survived": int(df.Survived.sum()),
    "died": int((df.Survived == 0).sum()),
    "survival_rate": r(df.Survived.mean() * 100),
    "avg_age": r(df.Age.mean()), "median_fare": r(df.Fare.median()),
    "pct_female": r((df.Sex == "female").mean() * 100),
    "pct_alone": r(df.IsAlone.mean() * 100),
    "avg_family": r(df.FamilySize.mean()),
    "by_sex": {"labels": list(sr("Sex").index), "rate": list(sr("Sex").values),
               "count": [int(x) for x in cnt("Sex").values]},
    "by_class": {"labels": list(sr("Pclass_name").index), "rate": list(sr("Pclass_name").values),
                 "count": [int(x) for x in cnt("Pclass_name").values]},
    "by_embark": {"labels": list(sr("Embarked_name").index), "rate": list(sr("Embarked_name").values),
                  "count": [int(x) for x in cnt("Embarked_name").values]},
    "by_age": {"labels": [str(x) for x in sr("AgeGroup").index], "rate": list(sr("AgeGroup").values),
               "count": [int(x) for x in cnt("AgeGroup").values]},
    "by_fareband": {"labels": [str(x) for x in sr("FareBand").index], "rate": list(sr("FareBand").values)},
    "by_family": {"labels": [str(x) for x in sr("FamilySize").index], "rate": list(sr("FamilySize").values),
                  "count": [int(x) for x in cnt("FamilySize").values]},
    "by_alone": {"labels": ["With Family", "Alone"], "rate": list(sr("IsAlone").values)},
    "by_title": {"labels": list(sr("Title").index), "rate": list(sr("Title").values),
                 "count": [int(x) for x in cnt("Title").values]},
    "by_cabin": {"labels": ["No Cabin", "Has Cabin"], "rate": list(sr("HasCabin").values)},
    "sex_class": {c: [r(x) for x in (df[df.Pclass_name == c].groupby("Sex")["Survived"].mean() * 100)]
                  for c in ["1st Class", "2nd Class", "3rd Class"]},
    "age_hist": {"s": np.histogram(df[df.Survived == 1].Age, bins=np.arange(0, 85, 5))[0].tolist(),
                 "d": np.histogram(df[df.Survived == 0].Age, bins=np.arange(0, 85, 5))[0].tolist(),
                 "bins": [f"{i}-{i+5}" for i in range(0, 80, 5)]},
    "fare_by_class": {c: r(df[df.Pclass_name == c].Fare.mean()) for c in ["1st Class", "2nd Class", "3rd Class"]},
    "scatter": [{"x": r(a), "y": r(f), "s": int(s)} for a, f, s in
                zip(df.Age, df.Fare_capped, df.Survived)],
    "corr": {c: r(df[c].corr(df.Survived)) for c in ["Pclass", "Age", "Fare", "FamilySize", "IsAlone", "HasCabin"]},
}
k["corr"]["Sex(female)"] = r((df.Sex == "female").astype(int).corr(df.Survived))

json.dump(k, open(OUT_JSON, "w"), indent=1)
{x: k[x] for x in ["survival_rate", "by_sex", "by_class", "corr", "outliers_fare"]}

## Key Insights

- **⚠️ `Survived` is not real data.** Every female passenger is marked survived (100%) and every male marked dead (0%) — the classic Kaggle `gender_submission` baseline, not actual outcomes. Any model or KPI built on this label is meaningless beyond "female = survived"; the other 12 columns are genuine and are what this analysis is really about.
- **Class buys a better cabin, not just a better label.** First class averages a noticeably higher fare than third class, and a cabin is recorded for far more first-class passengers — wealth shows up in `Fare` and `HasCabin`, which correlate with the label at 0.19 and 0.13.
- **Third class carried over half the passengers** (218 of 418) — any cross-class comparison has to account for this imbalance.
- **Age needed group-wise imputation.** A fifth of ages were missing; filling by `Title`+`Pclass` median keeps `Master`/`Miss` young and adults adult, avoiding the age-distribution flattening a single global median would cause.
- **Travelling alone is the strongest non-gender signal**, correlating with the label at -0.24 — solo travellers skew toward young third-class men.
- **Southampton dominates boarding**; Cherbourg passengers carry the highest average fare, entangling port with class.
- **Fare is heavily right-skewed**, with 55 fares above the IQR fence — capped for plotting, not deleted, since these are genuine first-class fares rather than data errors.


## 8. Interactive Chart Preview (Plotly)
One Plotly figure here as a sample — the full interactive, filterable dashboard is built with **Streamlit + Plotly** in `app_streamlit.py`, since Plotly figures alone are not "dashboards" (no filters/widgets) until wrapped in an app.

In [ ]:
import plotly.express as px

fig = px.scatter(df, x="Age", y="Fare_capped", color="Status",
                  hover_data=["Pclass_name", "Sex", "Title"],
                  title="Age vs Fare (interactive — hover, zoom, pan)",
                  color_discrete_map={"Survived": "#59A6A8", "Died": "#C9584A"})
fig.update_layout(template="plotly_white")
fig.show()

## Note on the labelled `Survived` column
Before drawing conclusions: `Survived` here is **100% determined by `Sex`** (every woman = survived, every man = died). This is the Kaggle *gender_submission* placeholder label, not a real outcome. It is kept in the analysis because it is the column the task asks about, but it should not be read as historical fact — the genuine signal in this dataset is in the other columns (class, fare, family size, cabin, age).